In [1]:
import os
import torch
import pandas as pd
from datasets import Dataset
from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, get_linear_schedule_with_warmup
from peft import get_peft_model, LoraConfig, TaskType, PeftModel
from torch.utils.data import DataLoader
from torch.optim import AdamW
from tqdm.notebook import tqdm

In [ ]:
os.environ["HF_TOKEN"] = "<HF TOKEN>"

In [ ]:
df = pd.read_parquet("/kaggle/input/datasets/abhinav10101010/medicalterms/Dataset.parquet")
print(df.shape)
print(df.columns.to_list())

(6861, 2)
['page_title', 'page_text']


In [4]:
df["word_count"] = df["page_text"].apply(lambda x: len(str(x).split()))

In [5]:
print(df["word_count"].describe())
print(f"\nTerms under 500 words:  {(df['word_count'] < 500).sum()}")
print(f"Terms 500–1500 words:   {(df['word_count'].between(500,1500)).sum()}")
print(f"Terms over 1500 words:  {(df['word_count'] > 1500).sum()}")

count     6861.000000
mean      1318.586941
std       1665.162402
min          0.000000
25%        307.000000
50%        754.000000
75%       1682.000000
max      24467.000000
Name: word_count, dtype: float64

Terms under 500 words:  2547
Terms 500–1500 words:   2364
Terms over 1500 words:  1950


In [6]:
MODEL_NAME        = "EleutherAI/pythia-1b"
MAX_LENGTH        = 512
BATCH_SIZE        = 8
EPOCHS            = 2
LR                = 2e-4
MAX_CHUNKS        = 20
CHUNK_SIZE        = 200
OVERLAP           = 50
SAVE_PATH         = "/kaggle/working/medical_lora_adapter"

In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Training on: {device}")

Training on: cuda


In [8]:
def chunk_text(text, chunk_size=200, overlap=50):
    words = text.split()

    if len(words) <= chunk_size:
        return [text]

    chunks = []
    start  = 0
    step   = chunk_size - overlap

    while start < len(words):
        end   = min(start + chunk_size, len(words))
        chunk = " ".join(words[start:end])

        if len(words[start:end]) >= overlap:
            chunks.append(chunk)

        start += step

    return chunks

In [9]:
MAX_CHUNKS_PER_TERM = 20

rows = []

for _, row in df.iterrows():
    term       = str(row["page_title"]).strip()
    definition = str(row["page_text"]).strip()

    if not term or not definition:
        continue

    chunks = chunk_text(definition, chunk_size=200, overlap=50)

    if len(chunks) > MAX_CHUNKS_PER_TERM:
        indices = [int(i * len(chunks) / MAX_CHUNKS_PER_TERM) for i in range(MAX_CHUNKS_PER_TERM)]
        chunks  = [chunks[i] for i in indices]

    for chunk in chunks:
        rows.append({"text": f"Term: {term}\nDescription: {chunk}\nEND"})
        rows.append({"text": f"Description: {chunk}\nDiagnosis: {term}\nEND"})

expanded_df = pd.DataFrame(rows)
print(f"Original rows:  {len(df)}")
print(f"Expanded rows:  {len(expanded_df)}")

samples_per_term = expanded_df.groupby(
    expanded_df["text"].str.extract(r"Term: (.+)\n")[0]
).size()
print(f"\nMax samples from one term:  {samples_per_term.max()}")
print(f"Avg samples per term:       {samples_per_term.mean():.1f}")

Original rows:  6861
Expanded rows:  102070

Max samples from one term:  60
Avg samples per term:       7.5


In [10]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME,trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/569 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/396 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

In [11]:
dataset = Dataset.from_pandas(expanded_df[["text"]])

def tokenize(batch):
    tokens = tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )
    tokens["labels"] = [
        [-100 if t == tokenizer.pad_token_id else t for t in ids]
        for ids in tokens["input_ids"]
    ]
    return tokens

tokenized = dataset.map(tokenize, batched=True, remove_columns=["text"])
tokenized.set_format("torch")
print(f"\nTokenized dataset: {tokenized}")

Map:   0%|          | 0/102070 [00:00<?, ? examples/s]


Tokenized dataset: Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 102070
})


In [12]:
split        = tokenized.train_test_split(test_size=0.1, seed=42)
train_loader = DataLoader(split["train"], batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(split["test"],  batch_size=BATCH_SIZE)

print(f"Train batches: {len(train_loader)}")
print(f"Val   batches: {len(val_loader)}")

Train batches: 11483
Val   batches: 1276


In [13]:
config = AutoConfig.from_pretrained(MODEL_NAME)
config.pad_token_id = config.eos_token_id
print("pad_token_id set to:", config.pad_token_id) 

pad_token_id set to: 0


In [14]:
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME,config=config)
model = model.to(device)

model.safetensors:   0%|          | 0.00/2.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

In [15]:
lora_config = LoraConfig(
    r=32,                    # higher rank than usual (16-32 works best for 1B models)
    lora_alpha=64,           # usually 2× rank
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["query_key_value", "dense", "dense_h_to_4h", "dense_4h_to_h"],
    modules_to_save=["embed_out"],  # optional but helps with medical tokens
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 119,799,808 || all params: 1,131,581,440 || trainable%: 10.5869


In [16]:
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)

In [17]:
scheduler = get_linear_schedule_with_warmup(
    optimizer, 
    num_warmup_steps=100, 
    num_training_steps=len(train_loader) * EPOCHS
)

In [18]:
for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0

    # Wrap train_loader with tqdm
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=True)

    for step, batch in enumerate(progress_bar):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )

        loss = outputs.loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_train_loss += loss.item()

        # Update the progress bar with live loss
        progress_bar.set_postfix(loss=f"{loss.item():.4f}")

    avg_train = total_train_loss / len(train_loader)
    print(f"\nEpoch {epoch+1} complete — Avg Train Loss: {avg_train:.4f}")

Epoch 1/2:   0%|          | 0/11483 [00:00<?, ?it/s]


Epoch 1 complete — Avg Train Loss: nan


Epoch 2/2:   0%|          | 0/11483 [00:00<?, ?it/s]


Epoch 2 complete — Avg Train Loss: nan


In [19]:
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
torch.save({
    'epoch': EPOCHS,
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict(),
}, f"{SAVE_PATH}/optimizer.pt")
print(f"Saved adapter to {SAVE_PATH}")

Saved adapter to /kaggle/working/medical_lora_adapter


In [20]:
model.eval()
total_val_loss = 0

with torch.no_grad():
        for batch in val_loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
            )
            total_val_loss += outputs.loss.item()

        avg_val = total_val_loss / len(val_loader)
print(f"\nEpoch {epoch+1} complete — Train loss: {avg_train:.4f} | Val loss: {avg_val:.4f}\n")



Epoch 2 complete — Train loss: nan | Val loss: nan



In [21]:
def query_model(prompt, max_new_tokens=40):
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    # Decode only the newly generated tokens, not the prompt
    generated = output[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)


# Test both directions
print("\n── Direction 1: Term → Description ──")
print(query_model("Term: Paracetamol Poisoning\nDescription:"))

print("\n── Direction 2: Symptoms → Diagnosis ──")
print(query_model("Description: patient presents with nausea and liver failure after overdose\nDiagnosis:"))

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.



── Direction 1: Term → Description ──


Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.




── Direction 2: Symptoms → Diagnosis ──

